# Project training configuration for local reproducible fine-tuning.

In [ ]:
import os
import re
import random
import unicodedata
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
BASE_MODEL_NAME = "indobenchmark/indobert-base-p1"
TRAIN_PATH = "./train_data/train.csv"
VAL_PATH = "./train_data/val.csv"
MODEL_OUTPUT_DIR = "./model"
TOKENIZER_OUTPUT_DIR = "./tokenizer"
MAX_LENGTH = 128
NUM_LABELS = 2
TEXT_COL = "text"
LABEL_COL = "Label"

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\zakia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


# PII masking utilities applied before preprocessing and tokenization.

In [2]:
PHONE_PATTERN = re.compile(r'(\+62|62|0)8[1-9][0-9]{6,10}')
EMAIL_PATTERN = re.compile(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}')
NIK_PATTERN = re.compile(r'\b\d{16}\b')

def mask_pii(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = EMAIL_PATTERN.sub("<EMAIL>", text)
    text = PHONE_PATTERN.sub("<PHONE>", text)
    text = NIK_PATTERN.sub("<NIK>", text)
    return text

# Core text normalization for Indonesian social-media comments.

In [5]:
from __future__ import annotations

import re
import unicodedata
from typing import Dict, Iterable, List, Tuple


# -----------------------------
# Emoji label mapping (selective)
# -----------------------------
_EMOJI_GROUPS: Dict[str, Tuple[str, ...]] = {
    "<MONEY>": ("💰", "🤑", "💸", "🏦", "💵", "💴", "💶", "💷", "🪙"),
    "<HYPE>": ("🔥", "🚀", "⚡", "💥", "✨", "⭐", "🌟"),
    "<OK>": ("✅", "✔", "☑"),
    "<WARN>": ("⚠", "🚨", "❗", "‼"),
    "<HEART>": ("❤️", "❤", "💖", "💗", "💓", "💞", "💘"),
    "<LAUGH>": ("😂", "🤣", "😹"),
    "<HAND>": ("👉", "👈", "👇", "👆", "🤝", "🙏"),
}

_EMOJI_TO_LABEL: Dict[str, str] = {
    emoji: label for label, emojis in _EMOJI_GROUPS.items() for emoji in emojis
}

_RE_ANY_LETTER = re.compile(r"[A-Za-z\u00C0-\u024F\u1E00-\u1EFF\u0400-\u04FF\u0600-\u06FF\u0900-\u097F\u4E00-\u9FFF]")
_RE_HAS_ALNUM = re.compile(r"[A-Za-z0-9]")
_RE_INVISIBLE = re.compile(r"[\u200B-\u200F\u202A-\u202E\u2060\uFEFF]")
_RE_COMBINING = re.compile(r"[\u0300-\u036F]")
_RE_WHITESPACE = re.compile(r"\s+")
_RE_NON_WORD_NO_TAG = re.compile(r"[^\w\s<>]")

_RE_TOKEN_TAG = re.compile(r"<[A-Z_]+>")
_RE_URL = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)

_RE_ASCII_ART_BLOCKS = re.compile(r"[▀▄█▌▐░▒▓■□◆◇●○◼◻◾◽╬═║╔╗╚╝╦╩╠╣╭╮╰╯┼─│]+")
_RE_LONG_SYMBOL_RUN = re.compile(r"([^\w\s<>])\1{2,}")

_RE_OBFUSCATED_SPACED = re.compile(r"\b(?:[A-Za-z]\s+){2,}[A-Za-z]\b")
_RE_OBFUSCATED_DOT = re.compile(r"\b(?:[A-Za-z]\.){2,}[A-Za-z]\b")
_RE_OBFUSCATED_MIXSEP = re.compile(r"\b(?:[A-Za-z][\.\-_/\\\|\s]){2,}[A-Za-z]\b")

_RE_TIMESTAMP_ONLY = re.compile(
    r"^\s*(?:\d{1,2}:)?\d{1,2}:\d{2}\s*$"
)

_RE_REPEATED_CHARS = re.compile(r"([a-z])\1{2,}", re.IGNORECASE)

_LEET_MAP = str.maketrans(
    {
        "0": "o",
        "1": "i",
        "3": "e",
        "4": "a",
        "5": "s",
        "6": "g",
        "7": "t",
        "8": "b",
        "9": "g",
        "@": "a",
        "$": "s",
    }
)


def unicode_nfkc(text: str) -> str:
    return unicodedata.normalize("NFKC", text)


def remove_invisible_chars(text: str) -> str:
    text = _RE_INVISIBLE.sub("", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Cf")
    return text


def fix_font_obfuscation(text: str) -> str:
    text = text.translate({0x00A0: 0x0020})
    text = _RE_COMBINING.sub("", text)
    return text


def emoji_to_labels(text: str) -> str:
    for emoji, label in _EMOJI_TO_LABEL.items():
        if emoji in text:
            text = text.replace(emoji, f" {label} ")
    return text

def is_timestamp_only(text: str) -> bool:
    if text is None:
        return False
    return bool(_RE_TIMESTAMP_ONLY.match(str(text)))

def remove_ascii_art_and_symbol_noise(text: str) -> str:
    text = _RE_ASCII_ART_BLOCKS.sub(" ", text)
    text = _RE_LONG_SYMBOL_RUN.sub(" ", text)
    return text


def _merge_obfuscated_word(match_text: str) -> str:
    merged = re.sub(r"[\s\.\-_/\\\|]+", "", match_text)
    return merged


def fix_separator_obfuscation(text: str) -> str:
    def merge_if_short_segments(m: re.Match) -> str:
        raw = m.group(0)
        segments = re.split(r"[\s\.\-_/\\\|]+", raw)
        segments = [s for s in segments if s]
        if not segments:
            return raw
        if all(len(s) <= 2 for s in segments) and len(segments) >= 3:
            return _merge_obfuscated_word(raw)
        return raw

    text = _RE_OBFUSCATED_MIXSEP.sub(merge_if_short_segments, text)
    return text


def normalize_leet_selective(token: str) -> str:
    digit_count = sum(c.isdigit() for c in token)
    alpha_count = sum(c.isalpha() for c in token)

    if digit_count >= 2:
        return token

    if len(token) > 5:
        return token

    if alpha_count == 0:
        return token

    return token.translate(_LEET_MAP)


def normalize_leetspeak(text: str) -> str:
    def repl_word(m: re.Match) -> str:
        w = m.group(0)
        return normalize_leet_selective(w)

    return re.sub(r"\b[\w@$]+\b", repl_word, text)


def normalize_repeated_chars(text: str) -> str:
    return _RE_REPEATED_CHARS.sub(r"\1\1", text)


def remove_meaningless_text(text: str) -> str:
    if not _RE_ANY_LETTER.search(text):
        return ""
    return text


def lowercase_text(text: str) -> str:
    return text.lower()


def cleanup_whitespace(text: str) -> str:
    text = _RE_WHITESPACE.sub(" ", text).strip()
    return text


def remove_symbol_noise_keep_tags(text: str) -> str:
    return _RE_NON_WORD_NO_TAG.sub(" ", text)


def preprocess_text(text: str) -> str:
    """
    Main preprocessing pipeline untuk satu komentar.
    Urutan mengikuti requirement:
    1) NFKC
    2) remove invisible
    3) remove ascii art / symbol noise
    4) emoji -> label
    5) remove meaningless text (tanpa huruf)
    6) fix separator / obfuscation
    7) normalize angka/simbol -> huruf (leetspeak)
    8) normalize repeated chars
    9) lowercase
    10) rapikan whitespace
    """
    if text is None:
        return ""

    text = str(text)

    if _RE_TIMESTAMP_ONLY.match(text):
        return ""
    text = unicode_nfkc(text)
    text = fix_font_obfuscation(text)
    text = remove_invisible_chars(text)

    text = emoji_to_labels(text)

    text = remove_ascii_art_and_symbol_noise(text)
    text = remove_symbol_noise_keep_tags(text)

    text = cleanup_whitespace(text)
    text = remove_meaningless_text(text)
    if not text:
        return ""

    text = fix_separator_obfuscation(text)
    text = normalize_leetspeak(text)
    text = normalize_repeated_chars(text)

    text = lowercase_text(text)
    text = cleanup_whitespace(text)

    return text

In [6]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

required_cols = {TEXT_COL, LABEL_COL}
if not required_cols.issubset(train_df.columns) or not required_cols.issubset(val_df.columns):
    raise ValueError(f"Dataset must contain columns: {required_cols}")

for df in [train_df, val_df]:
    df[TEXT_COL] = df[TEXT_COL].astype(str).apply(mask_pii).apply(preprocess_text)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

Train size: 6605
Val size: 826


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, local_files_only=False)

def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_ds = Dataset.from_pandas(train_df[[TEXT_COL, LABEL_COL]])
val_ds = Dataset.from_pandas(val_df[[TEXT_COL, LABEL_COL]])

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)

train_ds = train_ds.rename_column(LABEL_COL, "labels")
val_ds = val_ds.rename_column(LABEL_COL, "labels")

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=NUM_LABELS,
    local_files_only=False
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary")
    }

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=use_cuda
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
eval_result = trainer.evaluate()
print("Evaluation:", eval_result)

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(TOKENIZER_OUTPUT_DIR, exist_ok=True)

trainer.model.save_pretrained(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(TOKENIZER_OUTPUT_DIR)

print("Saved model to:", MODEL_OUTPUT_DIR)
print("Saved tokenizer to:", TOKENIZER_OUTPUT_DIR)